In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [2]:
df_ee = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/ee_2007_2015_bn.csv")

In [3]:
df_ee = df_ee.drop(["Unnamed: 0"], axis = 1)

In [4]:
df_ee

,UR,URFF,YUA,FDI,QOR,QOA,EIAP,PDPS,P65,P014,...,FEM,LFG,EPFRG,GGAG,PMSI,JI,DWCP,PR,DBPG,EUPC
0,4.59,3.76,10.10,13.51,3.78,4.99,4.78,32.0,17.37,14.85,...,73.842857,0.00,8.80,3732.0,56.666667,0.955,86.112980,90,0.0,5.509541e+07
1,5.45,5.14,12.03,8.12,4.00,5.10,3.90,32.0,17.44,14.89,...,73.842857,0.01,44.91,-2269.0,56.666667,0.955,86.252845,90,0.0,8.479186e+07
2,13.55,10.30,27.41,9.51,4.22,4.87,4.05,31.0,17.43,15.04,...,69.900000,-0.01,340.12,-3519.0,56.666667,0.955,85.403987,90,0.0,3.568504e+08
3,16.71,14.09,32.94,13.28,4.53,4.55,4.22,31.0,17.44,15.24,...,67.400000,-0.01,507.33,4528.0,56.666667,0.955,85.415456,80,0.0,3.969299e+08
4,12.33,11.58,22.39,4.80,4.49,4.39,4.41,31.0,17.57,15.43,...,68.600000,0.01,132.91,-34.0,56.666667,0.955,85.358224,80,0.0,1.797442e+08
5,10.02,9.10,20.86,7.69,4.22,4.46,4.49,31.0,17.86,15.59,...,71.100000,-0.01,285.06,-1474.0,56.666667,0.955,85.374812,80,0.0,4.990624e+08
6,8.63,8.15,18.70,4.31,4.20,4.14,4.26,30.0,18.20,15.74,...,71.600000,0.00,-239.99,1866.0,56.666667,0.955,85.428313,85,0.0,5.140841e+08
7,7.35,6.81,14.98,6.59,4.39,3.78,3.86,30.0,18.56,15.89,...,71.900000,0.00,169.83,-1231.0,56.666667,0.955,85.462140,90,0.0,2.871504e+08
8,6.41,6.32,14.34,-3.07,4.53,3.73,4.03,30.0,18.89,16.01,...,73.700000,0.00,173.78,-3206.0,56.666667,0.955,85.640301,90,0.0,6.873445e+07


In [5]:
ee = pd.DataFrame()
ee = df_ee.copy()
ee['EPFRG_1'] = ee['EPFRG'].shift(1)
ee['FDI_1'] = ee['FDI'].shift(1)
ee['EIS_1'] = ee['EIS'].shift(1)

In [6]:
ee = ee.dropna()

In [7]:
ee

,UR,URFF,YUA,FDI,QOR,QOA,EIAP,PDPS,P65,P014,...,GGAG,PMSI,JI,DWCP,PR,DBPG,EUPC,EPFRG_1,FDI_1,EIS_1
1,5.45,5.14,12.03,8.12,4.00,5.10,3.90,32.0,17.44,14.89,...,-2269.0,56.666667,0.955,86.252845,90,0.0,8.479186e+07,8.80,13.51,60.095527
2,13.55,10.30,27.41,9.51,4.22,4.87,4.05,31.0,17.43,15.04,...,-3519.0,56.666667,0.955,85.403987,90,0.0,3.568504e+08,44.91,8.12,60.831163
3,16.71,14.09,32.94,13.28,4.53,4.55,4.22,31.0,17.44,15.24,...,4528.0,56.666667,0.955,85.415456,80,0.0,3.969299e+08,340.12,9.51,64.503623
4,12.33,11.58,22.39,4.80,4.49,4.39,4.41,31.0,17.57,15.43,...,-34.0,56.666667,0.955,85.358224,80,0.0,1.797442e+08,507.33,13.28,65.465353
5,10.02,9.10,20.86,7.69,4.22,4.46,4.49,31.0,17.86,15.59,...,-1474.0,56.666667,0.955,85.374812,80,0.0,4.990624e+08,132.91,4.80,63.153093
6,8.63,8.15,18.70,4.31,4.20,4.14,4.26,30.0,18.20,15.74,...,1866.0,56.666667,0.955,85.428313,85,0.0,5.140841e+08,285.06,7.69,64.403281
7,7.35,6.81,14.98,6.59,4.39,3.78,3.86,30.0,18.56,15.89,...,-1231.0,56.666667,0.955,85.462140,90,0.0,2.871504e+08,-239.99,4.31,65.464427
8,6.41,6.32,14.34,-3.07,4.53,3.73,4.03,30.0,18.89,16.01,...,-3206.0,56.666667,0.955,85.640301,90,0.0,6.873445e+07,169.83,6.59,65.993575


In [8]:
sm = StructureModel()

In [9]:
sm.add_edges_from([
    ('EIS_1', 'EUPC'),
])

In [10]:
sm.edges

OutEdgeView([('EIS_1', 'EUPC')])

In [11]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_ee_2007_2015.html")

Graphs/fully_connected_ee_2007_2015.html


In [12]:
bn = BayesianNetwork(sm)

In [13]:
discretised_ee = pd.DataFrame(index=ee.index)

for col in ee.columns:
    no_unique = ee[col].nunique()
    
    if no_unique <= 1:
        discretised_ee[col] = 0
    else:
        try:
            discretised_ee[col] = pd.qcut(
                ee[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_ee[col] = ee[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_ee)

Discretised data:
   UR  URFF  YUA  FDI  QOR  QOA  EIAP  PDPS  P65  P014  ...  GGAG  PMSI  JI  \
1   0     0    0    2    0    2     0     2    0     0  ...     0     0   0   
2   2     2    2    2    0    2     1     1    0     0  ...     0     0   0   
3   2     2    2    2    2    2     1     1    0     0  ...     2     0   0   
4   2     2    2    0    2    1     2     1    1     1  ...     2     0   0   
5   1     1    1    1    0    1     2     1    1     1  ...     1     0   0   
6   1     1    1    0    0    0     2     0    2     2  ...     2     0   0   
7   0     0    0    1    1    0     0     0    2     2  ...     1     0   0   
8   0     0    0    0    2    0     0     0    2     2  ...     0     0   0   

   DWCP  PR  DBPG  EUPC  EPFRG_1  FDI_1  EIS_1  
1     2   1     0     0        0      2      0  
2     0   1     0     1        0      1      0  
3     1   0     0     2        2      2      1  
4     0   0     0     0        2      2      2  
5     0   0     0     2  

In [14]:
discretised_ee = discretised_ee.reset_index(drop=True)
bn.fit_node_states(discretised_ee)
baseline_auc = utils.get_avg_auc_all_info(discretised_ee, bn)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 3.030714750289917 seconds
Processing fold 1 using 7 cores takes 2.9916980266571045 seconds
Processing fold 2 using 7 cores takes 2.999100923538208 seconds
Processing fold 3 using 7 cores takes 2.972172975540161 seconds
Processing fold 4 using 7 cores takes 2.9213340282440186 seconds
Baseline AUC: 0.30625


In [15]:
edges_to_add = [('LV', 'EIS_1'), ('LV', 'EUPC')]
edges_to_remove = [('EIS_1', 'EUPC')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [16]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_ee_2007_2015.html")

Graphs/node_added_ee_2007_2015.html


In [17]:
discretised_ee['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_ee, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EIS_1' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 3.2675750255584717 seconds
Processing fold 1 using 7 cores takes 3.083696126937866 seconds
Processing fold 2 using 7 cores takes 3.0760929584503174 seconds
Processing fold 3 using 7 cores takes 3.0702900886535645 seconds
Processing fold 4 using 7 cores takes 3.1004650592803955 seconds
AUC from adding LV between 'EIS_1' and 'EUPC': 0.475
